In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\gagan\Desktop\Mtech AI ML\Sem 2\Natural Language Processing (S1-26_AIMLZG530)\Assignment\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch version: 2.14.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [4]:
%pip install nltk numpy pandas matplotlib

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 7.1 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.0 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.6 MB 8.8 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 11.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------------------- ------------------- 5.0/9.6 MB 23.9 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 26.5 MB/s eta 0:00:00
   -----------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import nltk
from nltk.corpus import treebank

In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [7]:
nltk.download("treebank")

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\treebank.zip.


True

In [8]:
print("Number of sentences:", len(treebank.sents()))
print("Number of words:", len(treebank.words()))
print("Number of tagged words:", len(treebank.tagged_words()))
print("Number of tagged sentences:", len(treebank.tagged_sents()))

Number of sentences: 3914
Number of words: 100676
Number of tagged words: 100676
Number of tagged sentences: 3914


In [9]:
sentences = treebank.sents()

print("Number of sentences:", len(sentences))
print("\nFirst sentence:")
print(sentences[0])

Number of sentences: 3914

First sentence:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']


In [11]:
num_sentences = len(sentences)
num_tokens = sum(len(sentence) for sentence in sentences)

print("Number of sentences:", num_sentences)
print("Total tokens:", num_tokens)
print("Average tokens per sentence:", num_tokens / num_sentences)

Number of sentences: 3914
Total tokens: 100676
Average tokens per sentence: 25.722023505365357


In [12]:
sentence_lengths = [len(sentence) for sentence in sentences]

print("Minimum sentence length:", min(sentence_lengths))
print("Maximum sentence length:", max(sentence_lengths))
print("Average sentence length:", np.mean(sentence_lengths))

Minimum sentence length: 1
Maximum sentence length: 271
Average sentence length: 25.722023505365357


In [14]:
## 4. Data Preprocessing
def preprocess_sentence(sentence):
    """
    Convert a Penn Treebank sentence into normalized tokens.
    """
    return [word.lower() for word in sentence]

sample_sentence = sentences[0]

print("Original:")
print(sample_sentence)

print("\nPreprocessed:")
print(preprocess_sentence(sample_sentence))

Original:
['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.']

Preprocessed:
['pierre', 'vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', '29', '.']


In [16]:
processed_sentences = [
    preprocess_sentence(sentence)
    for sentence in sentences
]

print("Number of processed sentences:", len(processed_sentences))
print(processed_sentences[0])

Number of processed sentences: 3914
['pierre', 'vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', '29', '.']


In [17]:
processed_sentences = [
    sentence for sentence in processed_sentences
    if len(sentence) > 0
]

print("Number of non-empty sentences:", len(processed_sentences))

Number of non-empty sentences: 3914


In [18]:
random.seed(SEED)

indices = list(range(len(processed_sentences)))
random.shuffle(indices)

split_idx = int(0.8 * len(indices))

train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

train_sentences = [processed_sentences[i] for i in train_indices]
test_sentences = [processed_sentences[i] for i in test_indices]

In [20]:
print("Training sentences:", len(train_sentences))
print("Testing sentences :", len(test_sentences))
# percentage
print("Training percentage:", len(train_sentences) / len(processed_sentences) * 100)
print("Testing percentage :", len(test_sentences) / len(processed_sentences) * 100)

Training sentences: 3131
Testing sentences : 783
Training percentage: 79.99489013796628
Testing percentage : 20.005109862033727


In [21]:
print("Sample training sentence:")
print(train_sentences[0])

print("\nSample test sentence:")
print(test_sentences[0])

Sample training sentence:
['the', 'company', 'noted', 'that', 'it', 'has', 'reduced', 'debt', 'by', '$', '1.6', 'billion', '*u*', 'since', 'the', 'end', 'of', '1988', 'and', 'bought', 'back', 'about', '15.5', 'million', 'shares', 'of', 'common', 'stock', 'since', 'the', 'fourth', 'quarter', 'of', '1987', '.']

Sample test sentence:
['terms', 'were', "n't", 'disclosed', '*-1', '.']


In [24]:
train_tokens = [
    token
    for sentence in train_sentences
    for token in sentence
]

test_tokens = [
    token
    for sentence in test_sentences
    for token in sentence
]

print("Training tokens:", len(train_tokens))
print("Testing tokens :", len(test_tokens))
print(train_tokens[:50])

Training tokens: 81021
Testing tokens : 19655
['the', 'company', 'noted', 'that', 'it', 'has', 'reduced', 'debt', 'by', '$', '1.6', 'billion', '*u*', 'since', 'the', 'end', 'of', '1988', 'and', 'bought', 'back', 'about', '15.5', 'million', 'shares', 'of', 'common', 'stock', 'since', 'the', 'fourth', 'quarter', 'of', '1987', '.', 'boeing', 'co.', 'said', '0', 'it', 'is', 'discussing', 'plans', '*ich*-1', 'with', 'three', 'of', 'its', 'regular', 'japanese']


In [25]:
unique_train_words = set(train_tokens)

print("Total training tokens:", len(train_tokens))
print("Unique training words:", len(unique_train_words))

Total training tokens: 81021
Unique training words: 10108


In [26]:
unique_test_words = set(test_tokens)

unseen_test_words = unique_test_words - unique_train_words

print("Unique test words:", len(unique_test_words))
print("Test words unseen during training:", len(unseen_test_words))

Unique test words: 4607
Test words unseen during training: 1279


In [27]:
print("=" * 50)
print("DATA PREPROCESSING SUMMARY")
print("=" * 50)

print(f"Total sentences      : {len(processed_sentences)}")
print(f"Training sentences   : {len(train_sentences)}")
print(f"Testing sentences    : {len(test_sentences)}")
print(f"Training tokens      : {len(train_tokens)}")
print(f"Testing tokens       : {len(test_tokens)}")
print(f"Unique train words   : {len(unique_train_words)}")
print(f"Unique test words    : {len(unique_test_words)}")
print(f"Unseen test words    : {len(unseen_test_words)}")

DATA PREPROCESSING SUMMARY
Total sentences      : 3914
Training sentences   : 3131
Testing sentences    : 783
Training tokens      : 81021
Testing tokens       : 19655
Unique train words   : 10108
Unique test words    : 4607
Unseen test words    : 1279


In [29]:
## 5. Vocabulary Construction

train_vocab_words = set(train_tokens)

print("Unique training words:", len(train_vocab_words))

Unique training words: 10108


In [30]:
vocab_words = sorted(train_vocab_words)

In [31]:
UNK_TOKEN = "<UNK>"

vocab_words = [UNK_TOKEN] + vocab_words

print("Vocabulary size:", len(vocab_words))

Vocabulary size: 10109


In [32]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(vocab_words)
}

In [33]:
print("Vocabulary size:", len(word_to_idx))

print("\nFirst 20 vocabulary entries:")
for word, idx in list(word_to_idx.items())[:20]:
    print(f"{word:15s} -> {idx}")

Vocabulary size: 10109

First 20 vocabulary entries:
<UNK>           -> 0
!               -> 1
#               -> 2
$               -> 3
%               -> 4
&               -> 5
'               -> 6
''              -> 7
'30s            -> 8
'40s            -> 9
'50s            -> 10
'80s            -> 11
'86             -> 12
'd              -> 13
'll             -> 14
'm              -> 15
're             -> 16
's              -> 17
've             -> 18
*               -> 19


In [34]:
idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}

In [35]:
for idx in range(10):
    print(idx, "->", idx_to_word[idx])

0 -> <UNK>
1 -> !
2 -> #
3 -> $
4 -> %
5 -> &
6 -> '
7 -> ''
8 -> '30s
9 -> '40s


In [36]:
print("UNK token ID:", word_to_idx[UNK_TOKEN])

UNK token ID: 0


In [37]:
def word_to_id(word):
    return word_to_idx.get(word, word_to_idx[UNK_TOKEN])

In [38]:
word_to_id("the")

9219

In [39]:
word_to_id("some_unknown_word")

0

In [40]:
test_word = train_tokens[0]

print("Word:", test_word)
print("ID:", word_to_id(test_word))

Word: the
ID: 9219


In [41]:
unknown_word = "thisworddoesnotexist123"

print("Word:", unknown_word)
print("ID:", word_to_id(unknown_word))

Word: thisworddoesnotexist123
ID: 0


In [42]:
def sentence_to_ids(sentence):
    return [word_to_id(word) for word in sentence]

In [43]:
sample_sentence = train_sentences[0]

print("Words:")
print(sample_sentence)

print("\nIDs:")
print(sentence_to_ids(sample_sentence))

Words:
['the', 'company', 'noted', 'that', 'it', 'has', 'reduced', 'debt', 'by', '$', '1.6', 'billion', '*u*', 'since', 'the', 'end', 'of', '1988', 'and', 'bought', 'back', 'about', '15.5', 'million', 'shares', 'of', 'common', 'stock', 'since', 'the', 'fourth', 'quarter', 'of', '1987', '.']

IDs:
[9219, 2811, 6565, 9218, 5389, 4845, 7703, 3284, 2334, 3, 463, 2064, 400, 8488, 9219, 3894, 6631, 680, 1568, 2172, 1864, 1278, 587, 6191, 8384, 6631, 2803, 8835, 8488, 9219, 4469, 7510, 6631, 678, 408]


In [44]:
test_ids = [
    sentence_to_ids(sentence)
    for sentence in test_sentences
]

In [45]:
unseen_count = 0
total_test_words = 0

for sentence in test_sentences:
    for word in sentence:
        total_test_words += 1

        if word not in word_to_idx:
            unseen_count += 1

print("Total test words:", total_test_words)
print("Unseen test words:", unseen_count)
print("Unseen percentage:",
      100 * unseen_count / total_test_words)

Total test words: 19655
Unseen test words: 1368
Unseen percentage: 6.960061053167133


In [46]:
from collections import Counter

word_counts = Counter(train_tokens)

print("Most common 20 words:")
for word, count in word_counts.most_common(20):
    print(f"{word:15s} : {count}")

Most common 20 words:
,               : 3947
the             : 3864
.               : 3062
of              : 1873
to              : 1756
a               : 1584
in              : 1415
and             : 1263
0               : 909
*-1             : 902
*               : 787
's              : 693
that            : 682
for             : 678
*t*-1           : 650
*u*             : 604
$               : 582
``              : 567
''              : 551
is              : 548


In [47]:
print("=" * 50)
print("VOCABULARY SUMMARY")
print("=" * 50)

print(f"Training tokens       : {len(train_tokens):,}")
print(f"Unique training words : {len(train_vocab_words):,}")
print(f"Vocabulary size       : {len(word_to_idx):,}")
print(f"UNK token             : {UNK_TOKEN}")
print(f"UNK token ID          : {word_to_idx[UNK_TOKEN]}")
print(f"Unseen test words     : {unseen_count:,}")
print(f"Unseen test %         : {100 * unseen_count / total_test_words:.2f}%")

VOCABULARY SUMMARY
Training tokens       : 81,021
Unique training words : 10,108
Vocabulary size       : 10,109
UNK token             : <UNK>
UNK token ID          : 0
Unseen test words     : 1,368
Unseen test %         : 6.96%


In [48]:
## 6. Sequence Generation
CONTEXT_SIZE = 3
def generate_sequences(sentences, word_to_idx, context_size=3):
    """
    Generate input-target pairs.

    Input:
        context_size preceding words

    Target:
        next word
    """
    
    X = []
    y = []

    for sentence in sentences:
        
        # Skip sentences that are too short
        if len(sentence) <= context_size:
            continue
        
        # Generate sliding windows
        for i in range(len(sentence) - context_size):
            
            context = sentence[i:i + context_size]
            target = sentence[i + context_size]
            
            # Convert words to integer IDs
            context_ids = [
                word_to_idx.get(word, word_to_idx["<UNK>"])
                for word in context
            ]
            
            target_id = word_to_idx.get(
                target,
                word_to_idx["<UNK>"]
            )
            
            X.append(context_ids)
            y.append(target_id)

    return X, y

In [49]:
X_train, y_train = generate_sequences(
    train_sentences,
    word_to_idx,
    context_size=CONTEXT_SIZE
)

In [50]:
print("Number of training sequences:", len(X_train))
print("Number of training targets:", len(y_train))

Number of training sequences: 71645
Number of training targets: 71645


In [52]:
X_test, y_test = generate_sequences(
    test_sentences,
    word_to_idx,
    context_size=CONTEXT_SIZE
)

print("Number of test sequences:", len(X_test))
print("Number of test targets:", len(y_test))

Number of test sequences: 17310
Number of test targets: 17310


In [53]:
print("First input IDs:")
print(X_train[0])

print("\nFirst target ID:")
print(y_train[0])

First input IDs:
[9219, 2811, 6565]

First target ID:
9218


In [54]:
first_context_words = [
    idx_to_word[idx]
    for idx in X_train[0]
]

first_target_word = idx_to_word[y_train[0]]

print("Context:", first_context_words)
print("Target :", first_target_word)

Context: ['the', 'company', 'noted']
Target : that


In [55]:
sample_data = []

for i in range(10):
    context_words = [
        idx_to_word[idx]
        for idx in X_train[i]
    ]
    
    target_word = idx_to_word[y_train[i]]
    
    sample_data.append({
        "Context": " ".join(context_words),
        "Target": target_word
    })

sample_df = pd.DataFrame(sample_data)

sample_df

,Context,Target
0,the company noted,that
1,company noted that,it
2,noted that it,has
3,that it has,reduced
4,it has reduced,debt
5,has reduced debt,by
6,reduced debt by,$
7,debt by $,1.6
8,by $ 1.6,billion
9,$ 1.6 billion,*u*


In [56]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

In [57]:
print("X_train shape:", X_train_tensor.shape)
print("y_train shape:", y_train_tensor.shape)

print("X_test shape :", X_test_tensor.shape)
print("y_test shape :", y_test_tensor.shape)

X_train shape: torch.Size([71645, 3])
y_train shape: torch.Size([71645])
X_test shape : torch.Size([17310, 3])
y_test shape : torch.Size([17310])


In [58]:
print("Input tensor:")
print(X_train_tensor[0])

print("\nTarget tensor:")
print(y_train_tensor[0])

Input tensor:
tensor([9219, 2811, 6565])

Target tensor:
tensor(9218)


In [59]:
print("\nDecoded input:")
print([
    idx_to_word[idx.item()]
    for idx in X_train_tensor[0]
])

print("\nDecoded target:")
print(idx_to_word[y_train_tensor[0].item()])


Decoded input:
['the', 'company', 'noted']

Decoded target:
that


In [60]:
assert X_train_tensor.shape[1] == 3
assert X_test_tensor.shape[1] == 3

assert len(X_train_tensor) == len(y_train_tensor)
assert len(X_test_tensor) == len(y_test_tensor)

print("All sequence checks passed!")

All sequence checks passed!


In [61]:
print("=" * 50)
print("SEQUENCE GENERATION SUMMARY")
print("=" * 50)

print(f"Context size       : {CONTEXT_SIZE}")
print(f"Training sequences : {len(X_train_tensor):,}")
print(f"Testing sequences  : {len(X_test_tensor):,}")

print("\nTensor shapes:")
print(f"X_train: {X_train_tensor.shape}")
print(f"y_train: {y_train_tensor.shape}")
print(f"X_test : {X_test_tensor.shape}")
print(f"y_test : {y_test_tensor.shape}")

SEQUENCE GENERATION SUMMARY
Context size       : 3
Training sequences : 71,645
Testing sequences  : 17,310

Tensor shapes:
X_train: torch.Size([71645, 3])
y_train: torch.Size([71645])
X_test : torch.Size([17310, 3])
y_test : torch.Size([17310])


In [62]:
## 7. PyTorch Dataset and DataLoader
class NextWordDataset(Dataset):
    """
    PyTorch Dataset for next-word prediction.

    Each sample contains:
        Input  : 3 preceding word IDs
        Target : next word ID
    """

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [66]:
train_dataset = NextWordDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = NextWordDataset(
    X_test_tensor,
    y_test_tensor
)

print("Training samples:", len(train_dataset))
print("Testing samples :", len(test_dataset))



Training samples: 71645
Testing samples : 17310


In [67]:
sample_X, sample_y = train_dataset[0]

print("Input:", sample_X)
print("Target:", sample_y)

Input: tensor([9219, 2811, 6565])
Target: tensor(9218)


In [68]:
print("Input words:")
print([
    idx_to_word[idx.item()]
    for idx in sample_X
])

print("\nTarget word:")
print(idx_to_word[sample_y.item()])

Input words:
['the', 'company', 'noted']

Target word:
that


In [69]:
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [70]:
batch_X, batch_y = next(iter(train_loader))

print("Input batch shape :", batch_X.shape)
print("Target batch shape:", batch_y.shape)

Input batch shape : torch.Size([64, 3])
Target batch shape: torch.Size([64])


In [71]:
for i in range(5):

    context = [
        idx_to_word[idx.item()]
        for idx in batch_X[i]
    ]

    target = idx_to_word[batch_y[i].item()]

    print(f"Sample {i+1}")
    print("Context:", context)
    print("Target :", target)
    print("-" * 40)

Sample 1
Context: ['calif.', ',', 'and']
Target : a
----------------------------------------
Sample 2
Context: ['own', 'hands', '.']
Target : ''
----------------------------------------
Sample 3
Context: ['anti-programmers', 'have', 'proposed']
Target : several
----------------------------------------
Sample 4
Context: ['mr.', 'canepa', 'received']
Target : a
----------------------------------------
Sample 5
Context: ['eight-person', 'executive', 'committee']
Target : who
----------------------------------------


In [72]:
print("Number of training batches:", len(train_loader))
print("Number of testing batches :", len(test_loader))

Number of training batches: 1120
Number of testing batches : 271


In [73]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [75]:
print("=" * 50)
print("PYTORCH DATA PIPELINE SUMMARY")
print("=" * 50)

print(f"Training samples : {len(train_dataset):,}")
print(f"Testing samples  : {len(test_dataset):,}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Training batches : {len(train_loader):,}")
print(f"Testing batches  : {len(test_loader):,}")

batch_X, batch_y = next(iter(train_loader))

print("\nSample batch:")
print(f"Input shape      : {batch_X.shape}")
print(f"Target shape     : {batch_y.shape}")
print(f"Device           : {device}")

PYTORCH DATA PIPELINE SUMMARY
Training samples : 71,645
Testing samples  : 17,310
Batch size       : 64
Training batches : 1,120
Testing batches  : 271

Sample batch:
Input shape      : torch.Size([64, 3])
Target shape     : torch.Size([64])
Device           : cuda


In [76]:
assert batch_X.shape[1] == CONTEXT_SIZE
assert batch_y.dim() == 1

assert len(train_dataset) == len(X_train_tensor)
assert len(test_dataset) == len(X_test_tensor)

print("All PyTorch data pipeline checks passed!")

All PyTorch data pipeline checks passed!
